In [1]:
# Step 1: Load the data
import pandas as pd

# Load sentiment scores
scores_df = pd.read_csv("./datasetFolder/t4sa_text_sentiment.tsv", sep="\t", names=["TWID", "NEG", "NEU", "POS"], skiprows=1)

# Load tweets
tweets_df = pd.read_csv("./datasetFolder/raw_tweets_text.csv")

# Merge both on ID
merged_df = pd.merge(tweets_df, scores_df, left_on="id", right_on="TWID")

In [2]:
# Step 2: Assign label based on highest score
def get_label(row):
    scores = {"NEG": row["NEG"], "NEU": row["NEU"], "POS": row["POS"]}
    return max(scores, key=scores.get)

label_map = {"NEG": 0, "POS": 1, "NEU": 2}
merged_df["label"] = merged_df.apply(get_label, axis=1).map(label_map)

# Keep only text and label
final_df = merged_df[["text", "label"]]


In [3]:
# Step 3: Balance the dataset
balanced_df = final_df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(175000, random_state=42)
).reset_index(drop=True)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_1956\2573162022.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = final_df.groupby('label', group_keys=False).apply(


In [4]:
# Step 4: Clean the text
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'\@\w+|\#','', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

balanced_df['clean_text'] = balanced_df['text'].apply(clean_text)

In [5]:
# Step 5: TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(balanced_df['clean_text'])
y = balanced_df['label']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [6]:
# Step 6: Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

print("\nLogistic Regression Results:")
print(classification_report(y_test, y_pred_log))
print("Accuracy:", accuracy_score(y_test, y_pred_log))


Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95     35000
           1       0.97      0.94      0.95     35000
           2       0.94      0.96      0.95     35000

    accuracy                           0.95    105000
   macro avg       0.95      0.95      0.95    105000
weighted avg       0.95      0.95      0.95    105000

Accuracy: 0.9519809523809524


In [15]:
# Define your custom text
custom_text = ["what do you think?"]

# Clean the text using the same function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#','', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

cleaned_custom = [clean_text(t) for t in custom_text]

# Vectorize using the same fitted vectorizer
custom_vec = vectorizer.transform(cleaned_custom)

# Predict using the trained logistic regression model
pred_label = log_model.predict(custom_vec)[0]


# Map label back to string
label_map_reverse = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
print("Predicted Sentiment:", label_map_reverse[pred_label])


Predicted Sentiment: Neutral


In [16]:
import pickle

# Save the logistic regression model
with open('savedPackages/logistic_model_T4sa_.pkl', 'wb') as f:
    pickle.dump(log_model, f)

# Save the TF-IDF vectorizer
with open('savedPackages/tfidf_vectorizer_t4sa.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
